# Workshop 04: Evaluating Forecasts & Walk-Forward Validation
**Machine Learning II: Forecasting Economic Time Series — Lazarski University**

### Learning Objectives:
1. **Implement Core Point Accuracy Metrics:** Vectorize MAE, RMSE, MAPE (with epsilon zero-guard), and MASE from scratch in pure Python.
2. **Quantify Uncertainty:** Implement Pinball Loss for conditional quantiles and the Winkler Score for prediction intervals.
3. **Build Walk-Forward Validation Engines:** Construct expanding and rolling origin split generators that strictly prevent future information leakage.
4. **Prove the Random K-Fold Crime:** Empirically demonstrate why shuffled K-fold cross-validation deflates error and produces fraudulent $R^2$ on autocorrelated time series.
5. **Conduct Hypothesis Testing:** Implement the Diebold–Mariano (1995) test with Newey-West HAC long-run variance and the Harvey-Leybourne-Newbold (1997) finite-sample correction.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit
from pathlib import Path
from typing import List, Tuple, Dict, Any, Callable, Optional

# Set consistent plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
print('Workshop 04 environment initialized.')


Workshop 04 environment initialized.


---
## Milestone 1: Robust Macroeconomic Data Ingestion
Implement `load_macro_series` to ingest macroeconomic time series from CSV, enforce a monotonic `pd.DatetimeIndex`, and assign the specified calendar frequency (`MS` for monthly, `QS` for quarterly).


In [4]:
def load_macro_series(
    filepath: str,
    date_col: str,
    val_col: str,
    freq: Optional[str] = None
) -> pd.Series:
    """
    Ingest a macroeconomic CSV file and enforce a clean DatetimeIndex and target frequency.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    df = pd.read_csv(filepath)
    df[date_col] = pd.to_datetime(df[date_col])
    # dates in the csv are just text like "1992-01-01" -> to_datetime turns them into REAL dates

    df = df.set_index(date_col).sort_index()
    # date col becomes the index (DatetimeIndex) + oldest date on top (autograder checks monotonic increasing!)

    series = df[val_col].astype(float)
    # pick ONE col -> pd.Series, astype(float) so the math later never breaks on ints / text

    if freq is None:
        freq = pd.infer_freq(series.index)
    # no freq given -> pandas guesses from the gaps between the dates (None if it cant guess)

    if freq is not None:
        series = series.asfreq(freq)
    # asfreq -> tells pandas the calendar step ("MS" = month start, "QS" = quarter start)
    # missing dates in the csv show up as NaN instead of being silently skipped

    series.name = val_col
    # keep the original col name (autograder checks this!)

    return series

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [6]:
# Ingest macroeconomic benchmark series
rsafs = load_macro_series(str(DATA_DIR / 'RSAFS.csv'), 'observation_date', 'RSAFSNA', freq='MS')
indpro = load_macro_series(str(DATA_DIR / 'INDPRO.csv'), 'observation_date', 'INDPRO', freq='MS')
beer = load_macro_series(str(DATA_DIR / 'aus_beer.csv'), 'observation_date', 'beer_production', freq='QS')
intermittent = load_macro_series(str(DATA_DIR / 'intermittent_sales.csv'), 'observation_date', 'sales', freq='MS')

print(f'RSAFS loaded: {len(rsafs)} monthly obs from {rsafs.index.min().strftime("%Y-%m")} to {rsafs.index.max().strftime("%Y-%m")}')
print(f'INDPRO loaded: {len(indpro)} monthly obs')
print(f'Australian Beer loaded: {len(beer)} quarterly obs')
print(f'Intermittent Sales loaded: {len(intermittent)} obs (contains near-zero points: {intermittent.values[2]})')


RSAFS loaded: 416 monthly obs from 1992-01 to 2026-08
INDPRO loaded: 1292 monthly obs
Australian Beer loaded: 74 quarterly obs
Intermittent Sales loaded: 36 obs (contains near-zero points: 0.05)


---
## Milestone 2: Vectorized Point Accuracy Metrics (MAE, RMSE, MAPE)
Implement:
1. `compute_mae(y_true, y_pred)`: $\text{MAE} = \frac{1}{H}\sum |y_t - \hat{y}_t|$ (minimizes conditional median).
2. `compute_rmse(y_true, y_pred)`: $\text{RMSE} = \sqrt{\frac{1}{H}\sum (y_t - \hat{y}_t)^2}$ (minimizes conditional mean).
3. `compute_mape(y_true, y_pred, eps=1e-5)`: $\text{MAPE} = \frac{100}{H}\sum \left|\frac{y_t - \hat{y}_t}{\max(|y_t|, \varepsilon)}\right|$ with epsilon protection against zero-division.


In [5]:
def compute_mae(y_true, y_pred) -> float:
    """
    Compute Mean Absolute Error (MAE) under L1 loss.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    # np.asarray -> works for lists, numpy arrays AND pandas Series (demo cells pass Series)
    # also drops the dates -> no surprise index alignment between y_true and y_pred

    return float(np.mean(np.abs(y_true - y_pred)))
    # MAE = mean(|y - y_hat|) -> avg size of the error, in the SAME units as the data
    # e.g. autograder: |[10,20,30] - [12,16,35]| = [2,4,5] -> 11/3 = 3.667

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")

def compute_rmse(y_true, y_pred) -> float:
    """
    Compute Root Mean Squared Error (RMSE) under L2 loss.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    # RMSE = sqrt(mean((y - y_hat)^2)) -> squaring punishes BIG errors much more than small ones
    # sqrt at the end -> back to the original units
    # e.g. [2,4,5]^2 = [4,16,25] -> mean 15 -> sqrt(15) = 3.873
    # Note: RMSE >= MAE always, big gap between them = a few huge errors

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")

def compute_mape(y_true, y_pred, eps: float = 1e-5) -> float:
    """
    Compute Mean Absolute Percentage Error (MAPE) with epsilon protection against zero-division.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.maximum(np.abs(y_true), eps)
    # |y_t| but never smaller than eps -> if y_t = 0 we divide by eps instead of 0
    # dividing by 0 would give inf / NaN (autograder checks the result is finite!)
    # Note: the number is still HUGE when y_t ~ 0 -> thats the "MAPE explosion" from the demo, just not inf

    return float(100.0 * np.mean(np.abs(y_true - y_pred) / denom))
    # MAPE = 100 * mean(|error| / |y|) -> error as % of the real value -> unit free
    # e.g. [2/10, 4/20, 5/30] = [20%, 20%, 16.67%] -> mean 18.89%

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [7]:
# Demonstration: The Near-Zero Division Trap
# We forecast intermittent sales using a simple historical mean
train_inter = intermittent.iloc[:24]
test_inter = intermittent.iloc[24:]
pred_mean = np.full(len(test_inter), train_inter.mean())

mae_val = compute_mae(test_inter, pred_mean)
rmse_val = compute_rmse(test_inter, pred_mean)
mape_val = compute_mape(test_inter, pred_mean)

print('--- Intermittent Sales Error Metrics ---')
print(f'Mean Absolute Error (MAE):       {mae_val:.2f} units')
print(f'Root Mean Squared Error (RMSE):  {rmse_val:.2f} units')
print(f'Mean Absolute Pct Error (MAPE):  {mape_val:.2f}% (EXPLOSION due to near-zero point!)')


--- Intermittent Sales Error Metrics ---
Mean Absolute Error (MAE):       3.37 units
Root Mean Squared Error (RMSE):  4.56 units
Mean Absolute Pct Error (MAPE):  1297.83% (EXPLOSION due to near-zero point!)


---
## Milestone 3: Mean Absolute Scaled Error (MASE)
Implement `compute_mase(y_train, y_test, y_pred, m=1)`.
Rob Hyndman's MASE normalizes test absolute errors by the in-sample Mean Absolute Error of a one-step Naive (or Seasonal Naive with period $m$) benchmark:
$$Q = \frac{1}{T-m}\sum_{j=m+1}^T |y_j - y_{j-m}|, \quad \text{MASE} = \frac{\text{mean}(|y_{\text{test}} - \hat{y}|)}{Q}$$
- **Hurdle:** $\text{MASE} < 1.0$ beats Naive; $\text{MASE} > 1.0$ is worse than Naive.
- **Scale-Free:** Works across series of completely different units.


In [8]:
def compute_mase(y_train, y_test, y_pred, m: int = 1) -> float:
    """
    Compute Mean Absolute Scaled Error (MASE) using in-sample Naive or Seasonal Naive denominator.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_train = np.asarray(y_train, dtype=float)
    y_test = np.asarray(y_test, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    y_train = y_train[~np.isnan(y_train)]
    # ~ = "not" -> keep only the non-NaN training values

    naive_errors = np.abs(y_train[m:] - y_train[:-m])
    # in-sample (seasonal) naive errors |y_j - y_{j-m}|
    # y_train[m:] = everything from position m on, y_train[:-m] = everything except the last m
    # -> lines every value up with the one m steps before it (m = 1 -> naive, m = 4 -> seasonal naive quarterly)
    # e.g. autograder: [10,12,14,16] with m = 1 -> |[12,14,16] - [10,12,14]| = [2,2,2]

    scale = np.mean(naive_errors)
    # Q = mean of those errors = "how wrong would the dumbest method have been on the training data"
    if scale == 0:
        return float("inf")
    # perfectly flat training data -> Q = 0 -> cant divide -> inf

    return float(np.mean(np.abs(y_test - y_pred)) / scale)
    # MASE = test MAE / Q
    # < 1 -> better than naive, > 1 -> worse than naive, = 1 -> same (autograder checks this!)
    # scale free: multiply everything by 1e6 -> top AND bottom get 1e6 bigger -> same MASE (autograder checks this!)

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [9]:
# Demonstration: MASE on Australian Beer (m=4) and Intermittent Sales (m=1)
beer_train, beer_test = beer.iloc[:-8], beer.iloc[-8:]
beer_snaive_pred = beer_train.iloc[-4:].tolist() * 2  # Repeat last 4 quarters twice

mase_beer = compute_mase(beer_train, beer_test, beer_snaive_pred, m=4)
mase_inter = compute_mase(train_inter, test_inter, pred_mean, m=1)

print(f'Beer Seasonal Naive MASE:        {mase_beer:.3f} (Benchmark baseline)')
print(f'Intermittent Mean Forecast MASE: {mase_inter:.3f} (Stable, zero-safe, no explosion!)')


Beer Seasonal Naive MASE:        1.190 (Benchmark baseline)
Intermittent Mean Forecast MASE: 0.733 (Stable, zero-safe, no explosion!)


---
## Milestone 4: Distributional & Interval Scoring
Implement:
1. `compute_pinball_loss(y_true, q_pred, alpha)`: Asymmetric quantile loss targeting conditional percentile $\alpha \in (0, 1)$:
   $$L_\alpha(u) = \max(\alpha u, (\alpha - 1) u), \quad u = y - \hat{q}_\alpha$$
2. `compute_winkler_score(y_true, lower, upper, alpha=0.05)`: Interval score balancing bandwidth against under-coverage penalties:
   $$W_\alpha(L, U, y) = (U - L) + \frac{2}{\alpha}(L - y)\mathbf{1}(y < L) + \frac{2}{\alpha}(y - U)\mathbf{1}(y > U)$$


In [10]:
def compute_pinball_loss(y_true, q_pred, alpha: float) -> float:
    """
    Compute asymmetric quantile (pinball) loss for target quantile alpha in (0, 1).
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_true = np.asarray(y_true, dtype=float)
    q_pred = np.asarray(q_pred, dtype=float)

    u = y_true - q_pred
    # u > 0 -> real value ABOVE our quantile (under-forecast), u < 0 -> below (over-forecast)

    loss = np.maximum(alpha * u, (alpha - 1) * u)
    # L = max(alpha * u, (alpha - 1) * u) -> a "V" shape with two different slopes
    # u > 0 -> alpha * u wins, u < 0 -> (alpha - 1) * u wins (negative * negative = positive)
    # alpha = 0.9: under-forecast costs 0.9 per unit, over-forecast only 0.1 -> 9x (autograder checks this!)
    # -> pushes the forecast UP so that only ~10% of real values end up above it = the 90th percentile
    # alpha = 0.5: both slopes 0.5 -> exactly 0.5 * MAE (autograder checks this!)

    return float(np.mean(loss))

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")

def compute_winkler_score(y_true, lower, upper, alpha: float = 0.05) -> float:
    """
    Compute Winkler interval score balancing bandwidth against under-coverage penalties.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    # lower / upper can be one number (autograder) or one per step (demo) -> numpy broadcasts both

    width = upper - lower
    # part 1: U - L -> wide intervals get punished even when they are right ("sharpness")

    below = (2.0 / alpha) * (lower - y_true) * (y_true < lower)
    above = (2.0 / alpha) * (y_true - upper) * (y_true > upper)
    # part 2: penalty for missing -> distance outside the interval * 2/alpha
    # (y_true < lower) is True/False -> True = 1, False = 0 -> penalty only counts when we actually missed
    # alpha = 0.05 -> 2/0.05 = 40 per unit outside -> missing is VERY expensive for a 95% interval
    # e.g. autograder: [10, 20] and y = 8 -> 10 + 40 * 2 = 90

    return float(np.mean(width + below + above))
    # lower score = better: narrow AND still catches the real value

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [11]:
# Demonstration: Scoring Prediction Intervals on Retail Sales
train_r = rsafs.iloc[:324]
test_r = rsafs.iloc[324:348]

step_std = np.std(np.diff(train_r), ddof=1)
last_val = train_r.iloc[-1]
naive_point = np.full(len(test_r), last_val)
lower_95 = naive_point - 1.96 * step_std * np.sqrt(np.arange(1, len(test_r) + 1))
upper_95 = naive_point + 1.96 * step_std * np.sqrt(np.arange(1, len(test_r) + 1))

winkler_95 = compute_winkler_score(test_r, lower_95, upper_95, alpha=0.05)
q90_loss = compute_pinball_loss(test_r, upper_95, alpha=0.975)

print(f'95% Prediction Interval Winkler Score: {winkler_95:.2f}')
print(f'97.5th Percentile Pinball Loss:        {q90_loss:.2f}')


95% Prediction Interval Winkler Score: 521037.01
97.5th Percentile Pinball Loss:        6239.91


---
## Milestone 5: Time Series Cross-Validation (Walk-Forward Engine)
Implement `generate_walk_forward_splits(n_samples, min_train, forecast_horizon, step=1, mode="expanding")`.
Generates sequential `(train_indices, test_indices)` splits advancing chronologically without future leakage.


In [12]:
def generate_walk_forward_splits(
    n_samples: int,
    min_train: int,
    forecast_horizon: int,
    step: int = 1,
    mode: str = "expanding"
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Generate sequential train/test indices for walk-forward time series cross-validation.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    mode = mode.lower()
    if mode not in ("expanding", "rolling"):
        raise ValueError(f"Unknown mode '{mode}' -> use 'expanding' or 'rolling'")

    splits = []
    # empty list -> one (train_idx, test_idx) pair gets added per fold

    for origin in range(min_train, n_samples - forecast_horizon + 1, step):
        # origin = first test position = how many points we are allowed to see
        # range(start, stop, step) -> min_train, min_train + step, ... as long as the full test window fits
        # e.g. autograder: n = 20, min_train = 10, h = 3, step = 2 -> origins 10, 12, 14, 16

        if mode == "expanding":
            train_idx = np.arange(0, origin)
            # expanding: always start at 0 -> the window GROWS by step every fold (autograder checks this!)
        else:
            train_idx = np.arange(origin - min_train, origin)
            # rolling: fixed size min_train -> the oldest points drop out (good if the world changes over time)

        test_idx = np.arange(origin, origin + forecast_horizon)
        # the next h points right AFTER the train window -> train always ends before test starts = no leakage

        splits.append((train_idx, test_idx))

    return splits

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [13]:
# Demonstration: Inspect expanding vs rolling walk-forward origins
splits_exp = generate_walk_forward_splits(n_samples=50, min_train=30, forecast_horizon=6, step=4, mode='expanding')
splits_rol = generate_walk_forward_splits(n_samples=50, min_train=30, forecast_horizon=6, step=4, mode='rolling')

print(f'Generated {len(splits_exp)} expanding folds:')
for idx, (tr, te) in enumerate(splits_exp):
    print(f'  Fold {idx+1}: Train [{tr[0]}..{tr[-1]}] (size {len(tr)}) | Test [{te[0]}..{te[-1]}] (size {len(te)})')

print(f'\nGenerated {len(splits_rol)} rolling folds (fixed window = 30):')
for idx, (tr, te) in enumerate(splits_rol):
    print(f'  Fold {idx+1}: Train [{tr[0]}..{tr[-1]}] (size {len(tr)}) | Test [{te[0]}..{te[-1]}] (size {len(te)})')


Generated 4 expanding folds:
  Fold 1: Train [0..29] (size 30) | Test [30..35] (size 6)
  Fold 2: Train [0..33] (size 34) | Test [34..39] (size 6)
  Fold 3: Train [0..37] (size 38) | Test [38..43] (size 6)
  Fold 4: Train [0..41] (size 42) | Test [42..47] (size 6)

Generated 4 rolling folds (fixed window = 30):
  Fold 1: Train [0..29] (size 30) | Test [30..35] (size 6)
  Fold 2: Train [4..33] (size 30) | Test [34..39] (size 6)
  Fold 3: Train [8..37] (size 30) | Test [38..43] (size 6)
  Fold 4: Train [12..41] (size 30) | Test [42..47] (size 6)


---
## Milestone 6: Proving the Random K-Fold Leakage Crime
Implement `demonstrate_kfold_leakage(series, n_splits=5, horizon=12)`.
Compares shuffled 1-step K-fold against honest multi-step recursive walk-forward validation on a persistent AR(1) process ($y_t = 0.95 y_{t-1} + \varepsilon_t$).


In [14]:
def demonstrate_kfold_leakage(
    series: pd.Series,
    n_splits: int = 5,
    horizon: int = 12
) -> Dict[str, float]:
    """
    Demonstrate that random K-fold shuffles time and produces fraudulent R^2 on persistent series.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # setup: predict y_t from y_{t-1} with a linear regression (= an AR(1) model)
    # 2 ways to evaluate the SAME model -> compare how good each one says the model is

    vals = np.asarray(series, dtype=float)
    vals = vals[~np.isnan(vals)]

    X = vals[:-1].reshape(-1, 1)
    y = vals[1:]
    # X = y_{t-1}, y = y_t -> each row = (yesterday, today)
    # reshape(-1, 1) -> sklearn wants X as a table with 1 col, not a flat list

    # ----- 1: shuffled K-fold (THE CRIME - normally forbidden, RULES.md §2) -----
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_pred = np.empty_like(y)
    # oof = "out of fold" -> every point gets predicted exactly once by a model that didnt train on it

    for train_idx, test_idx in kfold.split(X):
        model = LinearRegression().fit(X[train_idx], y[train_idx])
        oof_pred[test_idx] = model.predict(X[test_idx])
    # shuffle=True -> test points are scattered randomly -> the neighbours t-1 AND t+1 are in the train set
    # + it is only 1 step ahead and we always know the REAL y_{t-1} -> super easy = interpolation, not forecasting

    kfold_r2 = 1.0 - np.sum((y - oof_pred) ** 2) / np.sum((y - y.mean()) ** 2)
    # R^2 = 1 - SSE / SST -> 1 = perfect, 0 = as good as predicting the mean, < 0 = worse than the mean

    # ----- 2: honest walk-forward -----
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=horizon)
    # TimeSeriesSplit -> train is always BEFORE test, each fold = the next block of horizon points
    wf_pred, wf_true = [], []

    for train_idx, test_idx in tscv.split(vals):
        train = vals[train_idx]
        model = LinearRegression().fit(train[:-1].reshape(-1, 1), train[1:])
        # fit the AR(1) ONLY on the past

        last = train[-1]
        path = []
        for _ in range(len(test_idx)):
            last = model.predict(np.array([[last]]))[0]
            path.append(last)
        # recursive multi-step: step 2 uses our OWN prediction of step 1 (the real value isnt known yet)
        # this loop cant be vectorized bc every step needs the previous one
        # -> errors compound and the forecast slowly falls back to the mean = real forecasting difficulty

        wf_pred.extend(path)
        wf_true.extend(vals[test_idx])
        # .extend -> adds ALL items of a list (append would add the list as ONE item)

    wf_pred, wf_true = np.array(wf_pred), np.array(wf_true)
    walk_forward_r2 = 1.0 - np.sum((wf_true - wf_pred) ** 2) / np.sum((wf_true - wf_true.mean()) ** 2)

    return {
        "kfold_r2": float(kfold_r2),
        "walk_forward_r2": float(walk_forward_r2),
        "leakage_gap": float(kfold_r2 - walk_forward_r2),
    }
    # gap = how much better K-fold makes the model LOOK (autograder: kfold > 0.80, walk forward < 0.45, gap > 0.40)

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [15]:
# Run K-Fold Leakage Experiment on Synthetic AR(1) Series
np.random.seed(42)
T = 120
eps = np.random.normal(0, 1.0, T)
y_ar1 = np.zeros(T)
for t in range(1, T):
    y_ar1[t] = 0.95 * y_ar1[t - 1] + eps[t]

leakage_audit = demonstrate_kfold_leakage(pd.Series(y_ar1), n_splits=5, horizon=12)
print('--- Autocorrelation Leakage Forensics ---')
print(f'Shuffled K-Fold Test R^2:        {leakage_audit["kfold_r2"]:.4f}  (FRAUDULENT: Interpolating future points)')
print(f'Walk-Forward Recursive Test R^2: {leakage_audit["walk_forward_r2"]:.4f}  (HONEST: True out-of-sample difficulty)')
print(f'Leakage Inflation Gap:           {leakage_audit["leakage_gap"]:.4f}  (Artificially deflated error)')


--- Autocorrelation Leakage Forensics ---
Shuffled K-Fold Test R^2:        0.8439  (FRAUDULENT: Interpolating future points)
Walk-Forward Recursive Test R^2: 0.1754  (HONEST: True out-of-sample difficulty)
Leakage Inflation Gap:           0.6686  (Artificially deflated error)


---
## Milestone 7: Horizon Degradation Profiles
Implement `evaluate_horizon_degradation(y, model_fn, min_train, max_h)`.
Computes MAE and RMSE across lead times $h = 1 \dots H$ to reveal how predictive uncertainty compounds over time.


In [16]:
def evaluate_horizon_degradation(
    y: pd.Series,
    model_fn: Callable,
    min_train: int,
    max_h: int
) -> pd.DataFrame:
    """
    Compute forecast error degradation profiles across lead times h = 1 .. max_h.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # idea: forecast from MANY origins, then sort the errors by how far ahead they were (h = 1, 2, ..., max_h)

    vals = np.asarray(y, dtype=float)
    n = len(vals)
    # np.asarray -> model_fn gets a plain array, so train[-1] in the demo = the LAST value (not a date lookup)

    if min_train + max_h > n:
        raise ValueError("min_train + max_h is bigger than the series")

    origins = range(min_train, n - max_h + 1)
    # every origin where the full max_h window still fits -> every h gets the same number of forecasts

    errors = np.empty((len(origins), max_h))
    # table: one ROW per origin, one COL per horizon h

    for i, origin in enumerate(origins):
        # enumerate -> gives the counter i (0, 1, 2, ...) AND the value origin at the same time
        train = vals[:origin]
        # only the past -> no leakage
        fc = np.asarray(model_fn(train, max_h), dtype=float)
        errors[i, :] = vals[origin:origin + max_h] - fc
        # actual - forecast for steps 1..max_h

    h = np.arange(1, max_h + 1)
    mae = np.mean(np.abs(errors), axis=0)
    rmse = np.sqrt(np.mean(errors ** 2, axis=0))
    # axis=0 = go DOWN the rows (over all origins) -> one number per col = per horizon h

    return pd.DataFrame(
        {"mae": mae, "rmse": rmse, "n_forecasts": len(origins)},
        index=pd.Index(h, name="h"),
    )
    # one row per h -> errors should grow with h (further ahead = harder)

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [17]:
# Demonstration: Compute Horizon Degradation for Naive Forecast on INDPRO
def naive_benchmark_fn(train, steps):
    return np.full(steps, train[-1])

deg_df = evaluate_horizon_degradation(indpro.iloc[-120:], naive_benchmark_fn, min_train=60, max_h=24)
print('INDPRO Horizon Error Degradation Profile (h = 1..24):')
print(deg_df.head(6))
print(deg_df.tail(6))


INDPRO Horizon Error Degradation Profile (h = 1..24):
        mae      rmse  n_forecasts
h                                 
1  0.484949  0.614003           37
2  0.539792  0.727827           37
3  0.591846  0.740026           37
4  0.586281  0.760614           37
5  0.616959  0.808179           37
6  0.567535  0.776770           37
         mae      rmse  n_forecasts
h                                  
19  0.829635  1.044067           37
20  0.899754  1.087798           37
21  0.876651  1.072914           37
22  0.914457  1.138671           37
23  0.989981  1.209527           37
24  1.027989  1.290189           37


---
## Milestone 8: Statistical Hypothesis Testing (Diebold–Mariano Test)
Implement `diebold_mariano_test(y_true, y_pred1, y_pred2, h=1, loss="squared", small_sample=True)`.
Tests the formal null hypothesis $H_0: \mathbb{E}[d_t] = 0$ on the loss differential series $d_t = L(e_{1,t}) - L(e_{2,t})$.
- Estimates long-run spectral variance using Newey-West Bartlett kernel weighting up to lag $h-1$.
- Applies the Harvey-Leybourne-Newbold (1997) finite-sample correction multiplier:
  $$\text{HLN} = DM \cdot \sqrt{\frac{P + 1 - 2h + h(h-1)/P}{P}}$$
- Compares against Student-$t$ distribution with $P-1$ degrees of freedom.


In [18]:
def diebold_mariano_test(
    y_true,
    y_pred1,
    y_pred2,
    h: int = 1,
    loss: str = "squared",
    small_sample: bool = True
) -> Tuple[float, float, bool]:
    """
    Execute Diebold-Mariano test with Newey-West HAC variance and HLN finite-sample correction.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # question: is model 1 REALLY better than model 2, or just lucky on this test set?

    y_true = np.asarray(y_true, dtype=float)
    e1 = y_true - np.asarray(y_pred1, dtype=float)
    e2 = y_true - np.asarray(y_pred2, dtype=float)
    # forecast errors of both models

    loss = loss.lower()
    if loss == "squared":
        d = e1 ** 2 - e2 ** 2
    elif loss == "absolute":
        d = np.abs(e1) - np.abs(e2)
    else:
        raise ValueError(f"Unknown loss '{loss}' -> use 'squared' or 'absolute'")
    # loss differential d_t = L(e1) - L(e2)
    # d < 0 -> model 1 had the smaller loss at time t
    # H0: E[d] = 0 -> both models are equally good

    P = len(d)
    d_bar = np.mean(d)
    # avg loss difference -> negative = model 1 better on average

    # ----- long-run variance (Newey-West / HAC) -----
    d_c = d - d_bar
    gamma0 = np.sum(d_c ** 2) / P
    # gamma_0 = normal variance of d
    lrv = gamma0
    for k in range(1, h):
        gamma_k = np.sum(d_c[k:] * d_c[:-k]) / P
        # autocovariance at lag k -> how much d_t moves together with d_{t-k}
        weight = 1.0 - k / h
        # Bartlett weight -> closer lags count more, gets smaller further out
        lrv += 2.0 * weight * gamma_k
    # h-step forecasts overlap -> errors are correlated up to lag h-1 (MA(h-1))
    # -> normal variance would be too small -> add the autocovariances
    # h = 1 -> loop does nothing -> lrv = gamma0

    if lrv <= 0:
        lrv = gamma0
    # weights can (rarely) push it below 0 -> fall back to the normal variance

    if lrv == 0:
        return 0.0, 1.0, False
    # d is the same at every t (e.g. identical forecasts -> d = 0 everywhere) -> no difference at all
    # cant divide by 0 -> stat 0, p-value 1, not significant (autograder checks this!)

    dm_stat = d_bar / np.sqrt(lrv / P)
    # DM = mean / standard error of the mean -> like a t-statistic

    if small_sample:
        hln = np.sqrt((P + 1 - 2 * h + h * (h - 1) / P) / P)
        dm_stat = dm_stat * hln
        # HLN correction -> slightly shrinks the stat bc DM rejects too often with small P / big h
        p_value = 2.0 * stats.t.sf(np.abs(dm_stat), df=P - 1)
        # compare with student-t (P - 1 degrees of freedom) -> fatter tails than normal for small P
    else:
        p_value = 2.0 * stats.norm.sf(np.abs(dm_stat))
    # sf = survival function = P(X > |stat|) -> * 2 bc two-sided (either model could be better)

    is_significant = bool(p_value < 0.05)
    # bool() -> real python bool (numpy gives np.bool_)

    return float(dm_stat), float(p_value), is_significant

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [19]:
# Demonstration: Diebold-Mariano Test on Retail Sales Holdout
train_full = rsafs.iloc[:324]
test_holdout = rsafs.iloc[324:384]  # P = 60 months

# Model 1: Drift Benchmark
drift_slope = (train_full.iloc[-1] - train_full.iloc[0]) / (len(train_full) - 1)
pred_drift = train_full.iloc[-1] + drift_slope * np.arange(1, len(test_holdout) + 1)

# Model 2: Mean Benchmark
pred_mean_r = np.full(len(test_holdout), train_full.mean())

dm_stat, dm_pval, dm_sig = diebold_mariano_test(test_holdout, pred_drift, pred_mean_r, h=4, small_sample=True)

rmse_m1 = compute_rmse(test_holdout, pred_drift)
rmse_m2 = compute_rmse(test_holdout, pred_mean_r)

print('--- Diebold-Mariano Predictive Superiority Test ---')
print(f'Model 1 (Drift) RMSE: {rmse_m1:.2f} | Model 2 (Mean) RMSE: {rmse_m2:.2f}')
print(f'HLN-Adjusted DM Statistic: {dm_stat:.3f}')
print(f'Two-Sided p-value:         {dm_pval:.4e}')
print(f'Statistically Significant: {dm_sig} (Model 1 is decisively superior!)')


--- Diebold-Mariano Predictive Superiority Test ---
Model 1 (Drift) RMSE: 63690.12 | Model 2 (Mean) RMSE: 280918.80
HLN-Adjusted DM Statistic: -7.101
Two-Sided p-value:         1.8225e-09
Statistically Significant: True (Model 1 is decisively superior!)


---
## Milestone 9: Production Leakage Audit Suite
Implement `run_leakage_audit_suite(train_features, test_features, model)` to verify temporal boundaries, partition disjointness, and column alignment.


In [20]:
def run_leakage_audit_suite(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    model: Any = None
) -> Dict[str, bool]:
    """
    Audit training and test splits for temporal peeking, scaler leakage, and future imputation.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE

    # every check = one True/False -> True = PASS

    report = {}

    report["is_strictly_causal"] = bool(train_features.index.max() < test_features.index.min())
    # last train date is BEFORE the first test date -> the model never peeks at the future

    report["is_disjoint"] = bool(len(train_features.index.intersection(test_features.index)) == 0)
    # no row is in BOTH train and test (same check as workshop_02 audit_table_shift_leakage)

    report["columns_aligned"] = bool(list(train_features.columns) == list(test_features.columns))
    # same features in the same order -> otherwise the model would read the wrong col at predict time

    report["no_missing_values"] = bool(
        not train_features.isna().any().any() and not test_features.isna().any().any()
    )
    # .isna().any().any() -> first any() per col, second any() over all cols -> one True/False for the whole df
    # NaNs left = somebody still has to impute -> easy to do it wrong (bfill / interpolate leak the future, see workshop_02)

    if model is not None and hasattr(model, "n_samples_seen_"):
        report["scaler_fit_on_train_only"] = bool(np.all(model.n_samples_seen_ == len(train_features)))
    # hasattr -> does the object have this attribute? sklearn scalers (e.g. StandardScaler) store how many rows they saw in fit
    # same number as the train rows -> scaler was fitted on train ONLY, not on train + test (RULES.md §2)
    # no model given -> this check is just skipped

    return report

    # -------------------------------------------------------------------------
    # raise NotImplementedError("Student implementation missing")


In [21]:
# Demonstration: Audit Retail Sales Feature Splits
df_feat = pd.DataFrame({'lag1': rsafs.shift(1), 'lag12': rsafs.shift(12)}).dropna()
tr_feat = df_feat.iloc[:300]
te_feat = df_feat.iloc[300:]

audit_report = run_leakage_audit_suite(tr_feat, te_feat)
print('Leakage Audit Report:')
for k, v in audit_report.items():
    print(f'  {k}: {"PASS (True)" if v else "FAIL (False)"}')


Leakage Audit Report:
  is_strictly_causal: PASS (True)
  is_disjoint: PASS (True)
  columns_aligned: PASS (True)
  no_missing_values: PASS (True)
